# 🤖 AI Research Foundations - Module 12: Building GPT From Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amirlankalm/ai-research-foundations/blob/main/12_Building_GPT_From_Scratch/module_12.ipynb)

In this interactive notebook, we build and train a mini autoregressive Decoder-only Transformer (**NanoGPT**) from scratch in PyTorch, verify loss convergence below the uniform baseline, and generate text!

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set deterministic seeds
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Hand-Rolled Causal Multi-Head Self-Attention
No black-box library calls: query, key, and value linear projections, scaled dot-product attention, causal lower-triangular masking, and output projection.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, block_size: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.c_attn = nn.Linear(d_model, 3 * d_model, bias=False)
        self.c_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.d_model, dim=2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

print("CausalSelfAttention defined.")

## 2. Transformer Block & NanoGPT Model

In [ ]:
class Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, block_size: int, dropout: float = 0.0):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln_2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class NanoGPT(nn.Module):
    def __init__(self, vocab_size: int, block_size: int = 16, d_model: int = 48, n_heads: int = 3, n_layers: int = 2):
        super().__init__()
        self.block_size = block_size
        self.wte = nn.Embedding(vocab_size, d_model)
        self.wpe = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, block_size) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight # Weight tying

    def forward(self, idx, targets=None):
        B, T = idx.size()
        tok_emb = self.wte(idx)
        pos_emb = self.wpe(torch.arange(0, T, dtype=torch.long, device=idx.device))
        x = tok_emb + pos_emb
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

print("NanoGPT model class defined.")

## 3. Training & Validation on Text

In [ ]:
corpus = (
    "To be, or not to be, that is the question: "
    "Whether 'tis nobler in the mind to suffer "
    "The slings and arrows of outrageous fortune, "
    "Or to take arms against a sea of troubles "
    "And by opposing end them."
)
chars = sorted(list(set(corpus)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
data = torch.tensor([stoi[c] for c in corpus], dtype=torch.long)

n = int(0.8 * len(data))
train_data, val_data = data[:n], data[n:]
block_size = 16
model = NanoGPT(vocab_size=vocab_size, block_size=block_size, d_model=48, n_heads=3, n_layers=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

uniform_loss = -math.log(1.0 / vocab_size)
print(f"Vocab size: {vocab_size} | Uniform baseline loss: {uniform_loss:.4f}")

def get_batch(split, batch_size=4):
    d = train_data if split == "train" else val_data
    max_idx = len(d) - block_size - 1
    ix = torch.randint(0, max_idx, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

losses = []
for step in range(1, 151):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 30 == 0 or step == 1:
        xv, yv = get_batch("val")
        with torch.no_grad():
            _, val_loss = model(xv, yv)
        print(f"Step {step:03d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

print(f"Final train loss ({losses[-1]:.4f}) beat baseline ({uniform_loss:.4f}): {losses[-1] < uniform_loss}")

## 4. Autoregressive Text Generation

In [ ]:
@torch.no_grad()
def generate(model, prompt="To", max_new_tokens=40, temperature=0.8):
    model.eval()
    idx = torch.tensor([[stoi[c] for c in prompt]], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_token), dim=1)
    return "".join([itos[i] for i in idx[0].tolist()])

sample = generate(model, prompt="To", max_new_tokens=50)
print("Generated text:")
print(sample)